In [ ]:
import ifcopenshell
import pandas as pd

ifc_path = "../data/ifc/sample_model.ifc"

model = ifcopenshell.open(ifc_path)

print("IFC loaded successfully")

In [ ]:
element_types = [
    "IfcDoor",
    "IfcSpace",
    "IfcWall",
    "IfcSlab",
    "IfcSanitaryTerminal",
    "IfcWindow",
    "IfcStair",
    "IfcRamp"
]

for element_type in element_types:
    elements = model.by_type(element_type)
    print(element_type, ":", len(elements))

In [ ]:
spaces = model.by_type("IfcSpace")

space_data = []

for space in spaces:
    space_data.append({
        "GlobalId": space.GlobalId,
        "Name": space.Name,
        "LongName": getattr(space, "LongName", None),
        "Description": space.Description
    })

spaces_df = pd.DataFrame(space_data)

spaces_df.head(20)

In [ ]:
spaces_df

In [ ]:
wc_spaces = spaces_df[
    spaces_df["Name"].astype(str).str.lower().str.contains("wc|dda|shower", na=False)
    |
    spaces_df["LongName"].astype(str).str.lower().str.contains("wc|dda|shower", na=False)
]

wc_spaces

In [ ]:
sanitary = model.by_type("IfcSanitaryTerminal")

sanitary_data = []

for item in sanitary:
    sanitary_data.append({
        "GlobalId": item.GlobalId,
        "Name": item.Name,
        "ObjectType": getattr(item, "ObjectType", None),
        "PredefinedType": getattr(item, "PredefinedType", None),
        "Description": item.Description
    })

sanitary_df = pd.DataFrame(sanitary_data)
sanitary_df

In [ ]:
products = model.by_type("IfcProduct")

matches = []

keywords = ["wc", "toilet", "pan", "shower", "basin", "sink"]

for product in products:
    text = f"{product.is_a()} {getattr(product, 'Name', '')} {getattr(product, 'ObjectType', '')}"
    if any(k in text.lower() for k in keywords):
        matches.append({
            "IFC_Type": product.is_a(),
            "GlobalId": product.GlobalId,
            "Name": getattr(product, "Name", None),
            "ObjectType": getattr(product, "ObjectType", None)
        })

matches_df = pd.DataFrame(matches)
matches_df

In [ ]:
# Door Analysis

In [ ]:
doors = model.by_type("IfcDoor")

print("Number of doors:", len(doors))

In [ ]:
door_data = []

for door in doors:
    door_data.append({
        "GlobalId": door.GlobalId,
        "Name": door.Name,
        "Width": getattr(door, "OverallWidth", None),
        "Height": getattr(door, "OverallHeight", None),
        "ObjectType": getattr(door, "ObjectType", None)
    })

doors_df = pd.DataFrame(door_data)

doors_df

In [ ]:
MIN_ACCESSIBLE_DOOR_WIDTH = 775  # mm

doors_df["Door_Check_Status"] = doors_df["Width"].apply(
    lambda width: "PASS" if width >= MIN_ACCESSIBLE_DOOR_WIDTH else "FAIL"
)

doors_df[["Name", "Width", "Height", "Door_Check_Status"]]

In [ ]:
doors_df["Door_Check_Status"].value_counts()

In [ ]:
wc_dda_doors = doors_df[
    doors_df["Name"].astype(str).str.lower().str.contains("wc|dda", na=False)
    |
    doors_df["ObjectType"].astype(str).str.lower().str.contains("wc|dda", na=False)
]

wc_dda_doors[["Name", "Width", "Height", "ObjectType", "Door_Check_Status"]]

In [ ]:
doors_df.to_excel("../outputs/door_width_check.xlsx", index=False)
wc_dda_doors.to_excel("../outputs/wc_dda_door_width_check.xlsx", index=False)

In [ ]:
dda_doors = doors_df[
    doors_df["Name"].astype(str).str.lower().str.contains("dda", na=False)
    |
    doors_df["ObjectType"].astype(str).str.lower().str.contains("dda", na=False)
]

dda_doors[["Name", "Width", "Height", "ObjectType", "Door_Check_Status"]]

In [ ]:
wc_only_doors = doors_df[
    (
        doors_df["Name"].astype(str).str.lower().str.contains("wc", na=False)
        |
        doors_df["ObjectType"].astype(str).str.lower().str.contains("wc", na=False)
    )
    &
    ~(
        doors_df["Name"].astype(str).str.lower().str.contains("lab", na=False)
        |
        doors_df["ObjectType"].astype(str).str.lower().str.contains("lab", na=False)
    )
]

wc_only_doors[["Name", "Width", "Height", "ObjectType", "Door_Check_Status"]]

In [ ]:
doors_df.to_excel("../outputs/door_width_check.xlsx", index=False)
dda_doors.to_excel("../outputs/dda_door_width_check.xlsx", index=False)
wc_only_doors.to_excel("../outputs/wc_only_door_width_check.xlsx", index=False)

print("Door check outputs saved.")

In [ ]:
dda_spaces = spaces_df[
    spaces_df["Name"].astype(str).str.lower().str.contains("dda", na=False)
    |
    spaces_df["LongName"].astype(str).str.lower().str.contains("dda", na=False)
]

dda_spaces

In [ ]:
dda_spaces.to_excel("../outputs/dda_spaces.xlsx", index=False)

print("DDA spaces saved.")

In [ ]:
dda_spaces["Accessibility_Space_Type"] = dda_spaces["LongName"].apply(
    lambda x: "Accessible WC" if "wc" in str(x).lower()
    else "Accessible Shower Room" if "shower" in str(x).lower()
    else "Other Accessible Space"
)

dda_spaces

In [ ]:
def get_property_sets(element):
    property_sets = {}

    if hasattr(element, "IsDefinedBy"):
        for relation in element.IsDefinedBy:
            if relation.is_a("IfcRelDefinesByProperties"):
                prop_def = relation.RelatingPropertyDefinition

                if prop_def.is_a("IfcPropertySet"):
                    props = {}

                    for prop in prop_def.HasProperties:
                        if hasattr(prop, "NominalValue") and prop.NominalValue:
                            props[prop.Name] = prop.NominalValue.wrappedValue
                        else:
                            props[prop.Name] = None

                    property_sets[prop_def.Name] = props

    return property_sets

In [ ]:
for _, row in dda_spaces.iterrows():
    space = model.by_guid(row["GlobalId"])
    print("\nSPACE:", row["Name"], "-", row["LongName"])
    psets = get_property_sets(space)
    
    for pset_name, props in psets.items():
        print("\n", pset_name)
        for key, value in props.items():
            print("  ", key, ":", value)

In [ ]:
for _, row in dda_spaces.iterrows():
    space = model.by_guid(row["GlobalId"])
    
    print("\nSPACE:", row["Name"], "-", row["LongName"])
    
    if hasattr(space, "IsDefinedBy"):
        for relation in space.IsDefinedBy:
            if relation.is_a("IfcRelDefinesByProperties"):
                definition = relation.RelatingPropertyDefinition
                
                print("Definition type:", definition.is_a())
                print("Definition name:", getattr(definition, "Name", None))

In [ ]:
for item in sanitary:
    print("\nSANITARY:", item.Name)
    print("Type:", item.PredefinedType)
    
    if hasattr(item, "ContainedInStructure"):
        for rel in item.ContainedInStructure:
            print("Contained in:", rel.RelatingStructure.Name)

In [ ]:
for _, row in dda_spaces.iterrows():
    print("\nDDA SPACE:", row["Name"], "-", row["LongName"])

    space = model.by_guid(row["GlobalId"])

    if hasattr(space, "ContainsElements"):
        for rel in space.ContainsElements:
            for element in rel.RelatedElements:
                print(element.is_a(), "|", element.Name)

In [ ]:
dda_contained_elements = []

for _, row in dda_spaces.iterrows():
    space = model.by_guid(row["GlobalId"])

    if hasattr(space, "ContainsElements"):
        for rel in space.ContainsElements:
            for element in rel.RelatedElements:
                dda_contained_elements.append({
                    "Space_Name": row["Name"],
                    "Space_LongName": row["LongName"],
                    "IFC_Type": element.is_a(),
                    "Element_Name": element.Name
                })

dda_elements_df = pd.DataFrame(dda_contained_elements)

dda_elements_df

In [ ]:
dda_elements_df.to_excel("../outputs/dda_contained_elements.xlsx", index=False)

print("DDA contained elements saved.")

In [ ]:
required_keywords = {
    "Doc M toilet/shower pack": ["docm", "doc m"],
    "Grab rail": ["grabbar", "grab bar", "support rail", "suprt"],
    "Toilet roll holder": ["toiletroll", "tlt rll", "roll holder"],
    "Mirror": ["mirror"],
    "Soap dispenser": ["soap"],
    "Hand dryer": ["dryer"],
    "Coat hook": ["hook"]
}

dda_fixture_checks = []

for space_name in dda_elements_df["Space_LongName"].unique():
    elements_in_space = dda_elements_df[
        dda_elements_df["Space_LongName"] == space_name
    ]

    combined_text = " ".join(elements_in_space["Element_Name"].astype(str)).lower()

    for check_name, keywords in required_keywords.items():
        found = any(keyword.lower() in combined_text for keyword in keywords)

        dda_fixture_checks.append({
            "Space": space_name,
            "Check": check_name,
            "Status": "PASS" if found else "REVIEW"
        })

dda_fixture_checks_df = pd.DataFrame(dda_fixture_checks)

dda_fixture_checks_df

In [ ]:
dda_fixture_checks_df.to_excel("../outputs/dda_fixture_checks.xlsx", index=False)

print("DDA fixture checks saved.")